# LSTM / GRU: Time Series Classification

LSTM и GRU на 48h ценовых последовательностях Polymarket (14 фичей). Binary classification: направление цены через 12h.

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'

import torch
torch.set_num_threads(1)
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from torch.utils.data import Dataset, DataLoader

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DATA = Path('../../data/processed')
MODELS = Path('../../data/models')
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')

## Подготовка данных

1000 рынков, часовые цены. Features: price, return, volatility, momentum. Window=48h, horizon=12h.

In [ ]:
# Загрузка ценовых данных
prices = pd.read_parquet(DATA / 'prices.parquet')
print(f'Всего точек: {len(prices):,}')
print(f'Рынков: {prices.token_id.nunique()}')
print(f'Период: {prices.datetime.min()} → {prices.datetime.max()}')

# Длины рядов
seq_lens = prices.groupby('token_id').size()
print(f'\nДлина рядов: min={seq_lens.min()}, median={seq_lens.median():.0f}, max={seq_lens.max()}')
print(f'Рядов >= 100 точек: {(seq_lens >= 100).sum()}')

# Визуализация нескольких рынков
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
sample_tokens = prices.groupby('token_id').size().nlargest(6).index

for ax, token in zip(axes.flat, sample_tokens):
    ts = prices[prices.token_id == token].sort_values('datetime')
    ax.plot(ts.datetime, ts.price, linewidth=0.8)
    ax.set_title(f'len={len(ts)}', fontsize=9)
    ax.set_ylim(0, 1)
    ax.tick_params(labelsize=7)

plt.suptitle('Примеры ценовых рядов (6 рынков)', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Параметры задачи
WINDOW = 48       # смотрим на последние 48 часов (2 дня)
HORIZON = 12      # предсказываем направление через 12 часов
MIN_SEQ_LEN = WINDOW + HORIZON + 10  # минимальная длина ряда
N_FEATURES = 4    # price, return, volatility, momentum

def create_features(price_series):
    """Создаём фичи из ряда цен.
    
    Возвращает DataFrame с колонками:
    - price_norm: цена нормализованная в [0,1] (уже по сути так)
    - ret: часовой return (изменение цены)
    - vol: скользящая волатильность (std за 12 часов)
    - momentum: цена - SMA(12)
    """
    df = pd.DataFrame({'price': price_series.values})
    df['ret'] = df['price'].diff().fillna(0)
    df['vol'] = df['ret'].rolling(12, min_periods=1).std().fillna(0)
    df['momentum'] = df['price'] - df['price'].rolling(12, min_periods=1).mean()
    df['momentum'] = df['momentum'].fillna(0)
    return df[['price', 'ret', 'vol', 'momentum']].values

def create_sequences(prices_df, window=WINDOW, horizon=HORIZON):
    """Sliding window: для каждого рынка создаём пары (X, y).
    
    X: (window, n_features) — последовательность фичей за window часов
    y: 1 если цена через horizon часов > текущей, 0 иначе
    """
    all_X, all_y = [], []
    
    for token_id, group in prices_df.groupby('token_id'):
        ts = group.sort_values('datetime')
        if len(ts) < MIN_SEQ_LEN:
            continue
        
        features = create_features(ts['price'])
        prices_arr = ts['price'].values
        
        # Sliding window с шагом 6 (чтобы не было слишком много коррелированных примеров)
        for i in range(0, len(features) - window - horizon, 6):
            X = features[i : i + window]  # (window, n_features)
            current_price = prices_arr[i + window - 1]
            future_price = prices_arr[i + window + horizon - 1]
            y = 1.0 if future_price > current_price else 0.0
            
            all_X.append(X)
            all_y.append(y)
    
    return np.array(all_X, dtype=np.float32), np.array(all_y, dtype=np.float32)

print(f'Параметры: window={WINDOW}h, horizon={HORIZON}h, step=6h')
print(f'Минимальная длина ряда: {MIN_SEQ_LEN}')
print(f'Фичи: price, return, volatility, momentum')

X, y = create_sequences(prices)
print(f'\nDataset: X={X.shape}, y={y.shape}')
print(f'Class balance: UP={y.mean():.1%}, DOWN={1-y.mean():.1%}')

In [ ]:
# Time-based split (НЕ random — последовательности упорядочены по времени)
n = len(X)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')
print(f'Train UP%: {y_train.mean():.1%} | Val: {y_val.mean():.1%} | Test: {y_test.mean():.1%}')

# Нормализация фичей (fit на train, transform на val/test)
# Для каждой фичи: (x - mean) / std
train_mean = X_train.reshape(-1, N_FEATURES).mean(axis=0)
train_std = X_train.reshape(-1, N_FEATURES).std(axis=0) + 1e-8

X_train_norm = (X_train - train_mean) / train_std
X_val_norm = (X_val - train_mean) / train_std
X_test_norm = (X_test - train_mean) / train_std

print(f'\nНормализация (train stats):')
for i, name in enumerate(['price', 'return', 'volatility', 'momentum']):
    print(f'  {name}: mean={train_mean[i]:.4f}, std={train_std[i]:.4f}')

In [ ]:
# PyTorch Dataset и DataLoader
class TimeSeriesDataset(Dataset):
    """Dataset для временных рядов: X=(seq_len, features), y=binary"""
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 256

train_ds = TimeSeriesDataset(X_train_norm, y_train)
val_ds = TimeSeriesDataset(X_val_norm, y_val)
test_ds = TimeSeriesDataset(X_test_norm, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2)

# Проверяем shape
batch_X, batch_y = next(iter(train_loader))
print(f'Batch X: {batch_X.shape}  → (batch, seq_len={WINDOW}, features={N_FEATURES})')
print(f'Batch y: {batch_y.shape}  → (batch,)')
print(f'Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}')

## Baselines

In [ ]:
# Baselines на test set
np.random.seed(SEED)

# Momentum baseline: последний return > 0 → предсказываем UP
# X_test[:, -1, 1] = return на последнем шаге (фича idx=1)
momentum_pred = (X_test[:, -1, 1] > 0).astype(float)  # raw return > 0

# Mean reversion: последний return > 0 → предсказываем DOWN
mr_pred = (X_test[:, -1, 1] < 0).astype(float)

# Trend (среднее return за окно)
avg_return_pred = (X_test[:, :, 1].mean(axis=1) > 0).astype(float)

# Для AUC нужны вероятности, конвертируем через сигмоиду от momentum силы
momentum_prob = 1 / (1 + np.exp(-X_test[:, -6:, 1].sum(axis=1) * 50))  # сумма последних 6 returns
mr_prob = 1 - momentum_prob

results = {}
print(f'{"Baseline":<25} {"Accuracy":>10} {"AUC":>10}')
print('-' * 47)

for name, pred, prob in [
    ('Random', np.random.randint(0, 2, len(y_test)).astype(float), np.random.rand(len(y_test))),
    ('Always UP', np.ones(len(y_test)), np.ones(len(y_test)) * 0.5),
    ('Momentum (last ret)', momentum_pred, momentum_prob),
    ('Mean Reversion', mr_pred, mr_prob),
    ('Trend (avg return)', avg_return_pred, momentum_prob),
]:
    acc = accuracy_score(y_test, pred)
    try:
        auc = roc_auc_score(y_test, prob)
    except:
        auc = 0.5
    results[name] = {'acc': acc, 'auc': auc}
    print(f'{name:<25} {acc:>10.4f} {auc:>10.4f}')

## LSTM модель

LSTM(4→64) → Linear(64→32) → Linear(32→1). Dropout=0.3, batch_first=True, last hidden state.

In [ ]:
class PriceLSTM(nn.Module):
    """LSTM для предсказания направления цены."""
    
    def __init__(self, input_size=4, hidden_size=64, num_layers=1, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,       # input: (batch, seq_len, features)
            dropout=dropout if num_layers > 1 else 0,  # dropout между слоями LSTM
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        # x: (batch, seq_len, input_size)
        # output: (batch, seq_len, hidden_size) — все hidden states
        # (h_n, c_n): последние hidden state и cell state
        output, (h_n, c_n) = self.lstm(x)
        
        # h_n: (num_layers, batch, hidden_size) → берём последний слой
        last_hidden = h_n[-1]  # (batch, hidden_size)
        
        return self.classifier(last_hidden).squeeze(-1)
    
    def get_all_hidden_states(self, x):
        """Возвращает hidden states на каждом шаге (для визуализации)."""
        with torch.no_grad():
            output, _ = self.lstm(x)
            return output  # (batch, seq_len, hidden_size)

model = PriceLSTM(input_size=N_FEATURES, hidden_size=64).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'PriceLSTM: {total_params:,} параметров (все trainable)')
print(f'\nАрхитектура:')
print(model)

# Прогоним один batch для проверки
with torch.no_grad():
    test_out = model(batch_X.to(DEVICE))
    print(f'\nTest forward: input {batch_X.shape} → output {test_out.shape}')

In [ ]:
# Training utilities
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

@torch.no_grad()
def evaluate(model, loader):
    """Evaluate model on a DataLoader, return AUC and predictions."""
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        total_loss += loss.item() * len(y_batch)
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(y_batch.cpu().numpy())
    
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    avg_loss = total_loss / len(labels)
    auc = roc_auc_score(labels, preds)
    acc = accuracy_score(labels, (preds > 0.5).astype(int))
    return avg_loss, auc, acc, preds, labels

print('Training utilities ready.')

In [ ]:
# Training loop
EPOCHS = 20
best_val_auc = 0
patience_counter = 0
EARLY_STOP_PATIENCE = 5
history = {'train_loss': [], 'val_loss': [], 'val_auc': []}

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    n_samples = 0
    
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item() * len(y_batch)
        n_samples += len(y_batch)
    
    train_loss = total_loss / n_samples
    val_loss, val_auc, val_acc, _, _ = evaluate(model, val_loader)
    scheduler.step(val_loss)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    
    improved = val_auc > best_val_auc
    if improved:
        best_val_auc = val_auc
        torch.save(model.state_dict(), MODELS / 'price_lstm_v1.pt')
        patience_counter = 0
    else:
        patience_counter += 1
    
    lr = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | Train Loss: {train_loss:.4f} | '
          f'Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} | '
          f'Val Acc: {val_acc:.1%} | LR: {lr:.1e} {"★" if improved else ""}')
    
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f'\nEarly stopping: no improvement for {EARLY_STOP_PATIENCE} epochs')
        break

print(f'\nBest Val AUC: {best_val_auc:.4f}')

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss по эпохам')
axes[0].legend()

axes[1].plot(history['val_auc'], 'g-o', markersize=4, label='Val AUC')
axes[1].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUC')
axes[1].set_title('Validation AUC')
axes[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
# Test evaluation
model.load_state_dict(torch.load(MODELS / 'price_lstm_v1.pt', weights_only=True))
test_loss, test_auc, test_acc, test_preds, test_labels = evaluate(model, test_loader)

print(f'=== LSTM Test Results ===')
print(f'AUC:      {test_auc:.4f}')
print(f'Accuracy: {test_acc:.1%}')
print(f'Loss:     {test_loss:.4f}')
print(f'\nClassification Report:')
print(classification_report(test_labels, (test_preds > 0.5).astype(int),
                          target_names=['DOWN (0)', 'UP (1)']))

# Сохраняем в results
results['LSTM (1-layer)'] = {'acc': test_acc, 'auc': test_auc}

## Stacked LSTM, Bidirectional, GRU

In [ ]:
class StackedLSTM(nn.Module):
    """2-layer LSTM с dropout между слоями."""
    def __init__(self, input_size=4, hidden_size=64, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=2,           # 2 слоя!
            batch_first=True,
            dropout=dropout,        # dropout между LSTM слоями
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.classifier(h_n[-1]).squeeze(-1)


class BiLSTM(nn.Module):
    """Bidirectional LSTM — читает в обе стороны."""
    def __init__(self, input_size=4, hidden_size=64, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True,     # два направления!
        )
        # hidden_size * 2 — конкатенация forward и backward
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        # h_n: (2, batch, hidden) — forward и backward
        h_cat = torch.cat([h_n[0], h_n[1]], dim=1)  # (batch, hidden*2)
        return self.classifier(h_cat).squeeze(-1)


class PriceGRU(nn.Module):
    """GRU — упрощённый LSTM (2 гейта вместо 3)."""
    def __init__(self, input_size=4, hidden_size=64, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        _, h_n = self.gru(x)  # GRU не возвращает cell state
        return self.classifier(h_n[-1]).squeeze(-1)

# Сравнение количества параметров
for name, cls in [('Stacked LSTM (2-layer)', StackedLSTM), 
                   ('BiLSTM', BiLSTM), ('GRU', PriceGRU)]:
    m = cls(input_size=N_FEATURES)
    n_params = sum(p.numel() for p in m.parameters())
    print(f'{name:<25}: {n_params:>8,} params')

In [ ]:
# Обучаем все варианты и сравниваем
def train_model(model, name, epochs=15, lr=1e-3):
    """Train a model and return test metrics."""
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
    best_auc = 0
    no_improve = 0
    
    for epoch in range(epochs):
        model.train()
        total_loss, n = 0, 0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item() * len(y_b); n += len(y_b)
        
        val_loss, val_auc, val_acc, _, _ = evaluate(model, val_loader)
        sched.step(val_loss)
        
        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        
        if no_improve >= 5:
            break
    
    model.load_state_dict(best_state)
    test_loss, test_auc, test_acc, preds, labels = evaluate(model, test_loader)
    print(f'{name:<25} | Val AUC: {best_auc:.4f} | Test AUC: {test_auc:.4f} | Test Acc: {test_acc:.1%}')
    return test_auc, test_acc

print(f'{"Model":<25} | {"Val AUC":>9} | {"Test AUC":>9} | {"Test Acc":>9}')
print('-' * 63)

# Stacked LSTM
auc, acc = train_model(StackedLSTM(input_size=N_FEATURES), 'Stacked LSTM (2-layer)')
results['Stacked LSTM'] = {'acc': acc, 'auc': auc}

# BiLSTM
auc, acc = train_model(BiLSTM(input_size=N_FEATURES), 'BiLSTM')
results['BiLSTM'] = {'acc': acc, 'auc': auc}

# GRU
auc, acc = train_model(PriceGRU(input_size=N_FEATURES), 'GRU')
results['GRU'] = {'acc': acc, 'auc': auc}

## Hidden States визуализация

In [ ]:
# Визуализация hidden states для нескольких примеров
model.load_state_dict(torch.load(MODELS / 'price_lstm_v1.pt', weights_only=True))
model.eval()

# Берём 4 примера: 2 UP, 2 DOWN
up_idx = np.where(y_test == 1)[0][:2]
down_idx = np.where(y_test == 0)[0][:2]
sample_idx = np.concatenate([up_idx, down_idx])

fig, axes = plt.subplots(2, 4, figsize=(16, 6))

for col, idx in enumerate(sample_idx):
    seq = X_test[idx]  # (48, 4) — ненормализованный для визуализации
    seq_norm = torch.tensor(X_test_norm[idx:idx+1], dtype=torch.float32).to(DEVICE)
    
    # Hidden states
    hidden_states = model.get_all_hidden_states(seq_norm)[0].cpu().numpy()  # (48, 64)
    label = 'UP' if y_test[idx] == 1 else 'DOWN'
    pred = test_preds[idx]
    
    # Верхний ряд: ценовой ряд
    axes[0, col].plot(seq[:, 0], 'b-', linewidth=1.5)
    axes[0, col].set_title(f'True={label}, Pred={pred:.2f}', fontsize=9)
    axes[0, col].set_ylabel('Price' if col == 0 else '')
    
    # Нижний ряд: heatmap первых 16 hidden dimensions
    im = axes[1, col].imshow(hidden_states[:, :16].T, aspect='auto', cmap='RdBu_r')
    axes[1, col].set_xlabel('Timestep')
    axes[1, col].set_ylabel('Hidden dim' if col == 0 else '')

plt.suptitle('LSTM: ценовые ряды (верх) и hidden states (низ)', fontsize=12)
plt.tight_layout(); plt.show()
print('Hidden states меняются по мере обработки последовательности — LSTM накапливает информацию.')

In [ ]:
# Финальное сравнение всех моделей
print('=' * 55)
print(f'{"Model":<25} {"Accuracy":>12} {"AUC":>12}')
print('=' * 55)

# Сортируем по AUC
sorted_results = sorted(results.items(), key=lambda x: x[1]['auc'], reverse=True)
for name, metrics in sorted_results:
    print(f'{name:<25} {metrics["acc"]:>12.4f} {metrics["auc"]:>12.4f}')
print('=' * 55)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
names = [r[0] for r in sorted_results]
aucs = [r[1]['auc'] for r in sorted_results]
colors = ['#2ecc71' if 'LSTM' in n or 'GRU' in n or 'BiLSTM' in n else '#95a5a6' for n in names]

bars = ax.barh(names, aucs, color=colors)
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='Random (0.5)')
ax.set_xlabel('Test AUC')
ax.set_title('Сравнение моделей: предсказание направления цены')
ax.legend()

for bar, auc in zip(bars, aucs):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{auc:.4f}', va='center', fontsize=9)

plt.tight_layout(); plt.show()